# kf_area_total_merged.csv 인물 및 타이틀 중복 분석 노트북

통합된 역사 데이터셋인 `kf_area_total_merged.csv`에서 **인물(relate_prsn_nm)** 및 **타이틀(data_title_nm)** 의 중복 현황을 진단하고, 데이터 구조상의 특징과 중복 원인을 분석.

In [1]:
import os
import pandas as pd
import numpy as np

# 경로 설정
BASE_DIR = os.path.dirname(os.getcwd())
DATA_PATH = os.path.join(BASE_DIR, "data", "proceed", "kf_area_total_merged.csv")

print(f"분석 대상 파일 경로: {DATA_PATH}")

분석 대상 파일 경로: /Users/ahramkim/Documents/GitHub/k-heroes/backend/data/proceed/kf_area_total_merged.csv


### 1. 데이터 로드 및 기초 현황 확인

In [2]:
if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
    print("[SUCCESS] 데이터 로드 성공!")
    print(f" - 전체 행(Row) 수: {len(df):,}행")
    print(f" - 전체 열(Column) 수: {df.shape[1]}개")
else:
    print("[ERROR] 파일을 찾을 수 없습니다. 경로를 확인해주세요.")

[SUCCESS] 데이터 로드 성공!
 - 전체 행(Row) 수: 26,876행
 - 전체 열(Column) 수: 22개


### 2. 고유값(Unique Values) 통계

In [3]:
unique_titles = df["data_title_nm"].nunique()
unique_persons = df["relate_prsn_nm"].nunique()
unique_urls = df["cntnts_url"].nunique() if "cntnts_url" in df.columns else 0

print(f"- 고유 타이틀 (data_title_nm) 수: {unique_titles:,}개")
print(f"- 고유 인물 (relate_prsn_nm) 수: {unique_persons:,}개")
print(f"- 고유 콘텐츠 URL (cntnts_url) 수: {unique_urls:,}개")
print(f"- 타이틀 대비 평균 행 비율: {len(df) / unique_titles:.2f}배 (하나의 타이틀이 여러 행에 걸쳐 존재함)")

- 고유 타이틀 (data_title_nm) 수: 3,535개
- 고유 인물 (relate_prsn_nm) 수: 6,299개
- 고유 콘텐츠 URL (cntnts_url) 수: 3,511개
- 타이틀 대비 평균 행 비율: 7.60배 (하나의 타이틀이 여러 행에 걸쳐 존재함)


### 3. 완전 중복 행 확인

모든 컬럼의 데이터가 완전히 일치하는 행이 있는지 검사.

In [4]:
full_duplicates = df.duplicated().sum()
print(f"- 완전 중복 행 수: {full_duplicates}개")

- 완전 중복 행 수: 0개


### 4. 타이틀 (data_title_nm) 기준 중복 분석

타이틀이 중복되어 등장하는 빈도를 확인하고, 가장 많이 중복된 사례를 분석.

In [5]:
title_counts = df["data_title_nm"].value_counts()
dup_titles_count = (title_counts > 1).sum()

print(f"- 1회 초과 등장하는 중복 타이틀 수: {dup_titles_count:,}개")
print(f"- 전체 고유 타이틀 중 중복 타이틀 비율: {dup_titles_count / unique_titles * 100:.2f}%")
print("\n[가장 많이 중복된 타이틀 Top 10]")
print(title_counts.head(10))

- 1회 초과 등장하는 중복 타이틀 수: 2,656개
- 전체 고유 타이틀 중 중복 타이틀 비율: 75.13%

[가장 많이 중복된 타이틀 Top 10]
data_title_nm
판소리의 새로운 길을 모색한 얽매임 없는 예술가, 송만갑    177
판소리 이론을 세운 원조 명창, 김세종              158
일제강점기의 여류 명창, 배설향(裵雪香)             144
판소리의 집을 짓고 옷을 입힌 이, 신재효            143
국악의 사표가 된 명창, 만정 김소희               143
판소리의 성지 보성에서 열리는 서편제보성소리축제         128
불운한 시대의 천재 가객, 명창 임방울              128
동편제의 마지막 거장, 명창 박봉술                123
소리로 통정대부에 오른 명창 이동백                118
민중이 사랑한 명창 정정렬                     116
Name: count, dtype: int64


#### 타이틀 중복의 핵심 원인 분석

**인물(`relate_prsn_nm`) 또는 연관 이야기(`relate_stry_nm`)가 다르게 매핑** 되어 있음.

In [6]:
most_dup_title = title_counts.index[0]
most_dup_df = df[df["data_title_nm"] == most_dup_title]

print(f"타이틀: \'{most_dup_title}\'")
print(f"총 중복 행 수: {len(most_dup_df)}개")
print("\n처음 5개 행의 상세 정보 비교:")
display(most_dup_df[["data_manage_no", "data_manage_keyword", "relate_prsn_nm", "relate_stry_nm", "core_kwrd_cn"]].head(5))

타이틀: '판소리의 새로운 길을 모색한 얽매임 없는 예술가, 송만갑'
총 중복 행 수: 177개

처음 5개 행의 상세 정보 비교:


,data_manage_no,data_manage_keyword,relate_prsn_nm,relate_stry_nm,core_kwrd_cn
8,5,cltur,감상기,"민요를 채보하고 현대적으로 재건한 국악의 아버지, 지영희","순천, 전주대사습놀이, 조선성악연구회, 동편제"
96,49,cltur,강도근,"동편제의 마지막 거장, 명창 박봉술","순천, 전주대사습놀이, 조선성악연구회, 동편제"
352,177,cltur,강태흥,"가야금 산조의 창시자, 악성 김창조(金昌祖)","순천, 전주대사습놀이, 조선성악연구회, 동편제"
602,302,cltur,고수관,"귀신에게 직접 배운 듯한 귀곡성의 명수, 가왕 송흥록|바람처럼 떠돌던 중고제의 마지...","순천, 전주대사습놀이, 조선성악연구회, 동편제"
688,345,cltur,고유섭,"우리 미술사 정립이라는 독립운동의 횃불, 고유섭|한국미를 지킨 우리 시대 최고의 안...","순천, 전주대사습놀이, 조선성악연구회, 동편제"


#### 도메인(cltur vs prsn) 간 URL 중복 여부 확인

In [7]:
if "data_manage_keyword" in df.columns and "cntnts_url" in df.columns:
    cltur_urls = set(df[df["data_manage_keyword"] == "cltur"]["cntnts_url"].dropna())
    prsn_urls = set(df[df["data_manage_keyword"] == "prsn"]["cntnts_url"].dropna())
    common_urls = cltur_urls.intersection(prsn_urls)
    
    print(f"- cltur 도메인의 고유 URL 수: {len(cltur_urls):,}개")
    print(f"- prsn 도메인의 고유 URL 수: {len(prsn_urls):,}개")
    print(f"- 양쪽 모두에 존재하는 공통 URL 수: {len(common_urls):,}개")
    print(f"- cltur URL 중 prsn과 겹치는 비율: {len(common_urls) / len(cltur_urls) * 100:.2f}%")
else:
    print("필요한 컬럼(data_manage_keyword, cntnts_url)이 없습니다.")

- cltur 도메인의 고유 URL 수: 1,011개
- prsn 도메인의 고유 URL 수: 3,220개
- 양쪽 모두에 존재하는 공통 URL 수: 720개
- cltur URL 중 prsn과 겹치는 비율: 71.22%


### 5. 인물 (relate_prsn_nm) 기준 중복 분석

In [7]:
person_counts = df["relate_prsn_nm"].value_counts()
print("[가장 많이 등장하는 인물 Top 10]")
print(person_counts.head(10))

[가장 많이 등장하는 인물 Top 10]
relate_prsn_nm
고종     150
정조     114
송만갑    100
인조      95
한성준     85
송시열     83
허균      80
이광수     78
이승만     78
이황      78
Name: count, dtype: int64


#### 동일 타이틀 + 동일 인물 조합 (Pair) 중복 확인

실제로 `(data_title_nm, relate_prsn_nm)` 조합이 완벽히 중복되는 행이 존재하는지 분석.

In [ ]:
pair_cols = ["data_title_nm", "relate_prsn_nm"]
duplicate_pairs_mask = df.duplicated(subset=pair_cols, keep=False)
dup_pairs_df = df[duplicate_pairs_mask]

print(f"- 중복되는 타이틀+인물 조합을 가진 총 행(Row) 수: {len(dup_pairs_df):,}개")
print(f"- 중복되는 고유 타이틀+인물 쌍(Pair) 수: {dup_pairs_df.groupby(pair_cols).ngroups:,}개")

if len(dup_pairs_df) > 0:
    print("\n[중복 쌍 예시]")
    grouped = dup_pairs_df.groupby(pair_cols)
    for name, group in grouped:
        if len(group) >= 2:
            display(group[["data_manage_no", "data_title_nm", "relate_prsn_nm", "data_manage_keyword", "relate_stry_nm", "core_kwrd_cn"]])
            break

- 중복되는 타이틀+인물 조합을 가진 총 행(Row) 수: 2,536개
- 중복되는 고유 타이틀+인물 쌍(Pair) 수: 1,268개

[중복 쌍 예시]


,data_manage_no,data_title_nm,relate_prsn_nm,data_manage_keyword,relate_stry_nm,core_kwrd_cn
8103,4052,1920년대 최고의 예인 박춘재,박춘재,prsn,"조선의 마지막 광대, 예인 이동안|민요를 채보하고 현대적으로 재건한 국악의 아버지,...","박춘재, 판소리, 명창, 발탈, 장소팔, 재담"
10918,5460,1920년대 최고의 예인 박춘재,박춘재,cltur,"조선의 마지막 광대, 예인 이동안|민요를 채보하고 현대적으로 재건한 국악의 아버지,...","박춘재, 판소리, 명창, 발탈, 장소팔, 재담"
